# 실습 데이터분석가처럼 문제 정의하기

## 서울 공공자전거 대여 데이터

서울시 공공자전거 운영 담당자가 이렇게 말했습니다.

> "자전거가 언제, 어떤 때 많이 쓰이는지 알고 싶습니다."

이 말은 아직 **모호한 문제**입니다. 데이터를 직접 다루며 이 문제를 **분석할 수 있는 질문으로 바꾸고, 질문에 답하는 방법**을 익힙니다.

- 데이터 서울 공공자전거의 1년(365일) × 24시간 대여 기록, 8,760행 14열
- 출처 UCI Machine Learning Repository, Seoul Bike Sharing Demand (https://archive.ics.uci.edu/dataset/560/seoul+bike+sharing+demand)

### 구성

| 번호 | 하는 일 |
|:---|:---|
| 1 | 이론의 EDA 절차를 이 데이터로 그대로 수행합니다. (1.1~1.24) |
| 2 | **문제를 먼저 정의하고**, EDA로 데이터를 점검하며 분석 방법을 설계합니다. (2.1~2.5) |
| 3 | **질문 없이 탐색적 데이터 분석부터 수행하고**, 탐색에서 남은 의문으로 분석 질문을 도출합니다. (3.1~3.7) |

### 진행 방법

- 코드는 **직접** 작성합니다. 각 문항은 알아내야 할 것만 알려 줍니다. 쓸 도구는 스스로 정합니다.
- 코드 셀을 위에서 아래로 차례로 실행합니다. 뒤 문항은 앞 문항의 변수를 사용합니다.
- 코드를 쓰기 전에 **결과를 먼저 예상**합니다. 예상과 다르면 이유를 생각합니다.
- 글로 답하는 문항은 코드 셀 맨 위에 `#` 주석으로 쓰거나, 셀을 추가해 텍스트로 씁니다.
- 숫자만 쓰지 말고 **그 숫자가 무엇을 뜻하는지** 문장으로 씁니다.

### 쓸 수 있는 도구

| 하고 싶은 일 | 도구 |
|:---|:---|
| 데이터 미리보기 | `head()` `tail()` `sample()` |
| 데이터 구조 확인 | `shape` `columns` `dtypes` `info()` |
| Series, 위치 선택 | `pd.Series()` `iloc[ ]` `loc[ ]` `copy()` |
| 변수 선택, 필터링 | `df[["열"]]` `df[조건]` `&` `\|` `unique()` |
| 열 추가, 삭제 | `df["새 열"] = 계산` `drop()` |
| 정렬 | `sort_values()` |
| 결측값, 중복값 확인 | `isna()` `duplicated()` |
| 빈도 계산 | `value_counts()` `len()` |
| 기술통계 | `describe()` `mean()` `median()` |
| 그룹별 집계 | `groupby()` `agg()` `idxmax()` |
| 표 만들기 | `pd.DataFrame()` `round()` |

## 준비 데이터 로드

아래 두 셀은 **이미 작성되어 있습니다.** 실행만 하세요. 파일은 `data` 폴더 또는 노트북과 같은 폴더에서 찾습니다. Colab에서는 왼쪽 폴더 아이콘으로 `SeoulBikeData.csv`를 올려 두세요. 이 파일은 온도 기호(°) 때문에 `encoding="latin1"`로 읽어야 글자가 깨지지 않습니다.

In [ ]:
from pathlib import Path
import pandas as pd

candidates = [
    Path("data/SeoulBikeData.csv"),
    Path("강의교재/data/SeoulBikeData.csv"),
    Path("SeoulBikeData.csv"),
]

data_path = next((p for p in candidates if p.exists()), None)
if data_path is None:
    raise FileNotFoundError("SeoulBikeData.csv 파일을 찾을 수 없습니다. 파일을 올린 뒤 다시 실행하세요.")

bike = pd.read_csv(data_path, encoding="latin1")
print("불러온 파일:", data_path)

In [ ]:
# 원본의 긴 열 이름을 짧게 바꿉니다. (원본 파일은 바뀌지 않습니다.)
bike = bike.rename(columns={
    "Date": "date",
    "Rented Bike Count": "rented_count",
    "Hour": "hour",
    "Temperature(°C)": "temperature",
    "Humidity(%)": "humidity",
    "Wind speed (m/s)": "wind_speed",
    "Visibility (10m)": "visibility",
    "Dew point temperature(°C)": "dew_point",
    "Solar Radiation (MJ/m2)": "solar_radiation",
    "Rainfall(mm)": "rainfall",
    "Snowfall (cm)": "snowfall",
    "Seasons": "season",
    "Holiday": "holiday",
    "Functioning Day": "functioning_day",
})
print(bike.columns.tolist())

# 1. EDA 절차 따라 하기

분석 질문이 하나 주어졌습니다. 이 질문에 답하는 과정을 이론과 같은 순서로 밟습니다.

> 서울 공공자전거의 시간대별 평균 대여량은 어떻게 다르며, 어느 시간대에 대여가 가장 많은가?

### 1.1 분석 질문에 필요한 변수 정의

데이터를 열기 전에 먼저 생각합니다. ① 이 질문에 **필요한 변수(열)** ② **필요 없는 변수** ③ **집계 방법**을 쓰세요. 그리고 ④ 분석을 시작하기 전에 데이터가 이 질문에 답할 수 있는지 **점검해야 할 항목 네 가지**를 쓰세요.

### 1.2 변수의 종류 분류

`bike`의 열 14개를 **범주형**과 **연속형**으로 분류하세요. 숫자로 저장되어 있지만 범주처럼 다루는 열이 있다면 이유와 함께 쓰세요. 이 분석 질문(시간대별 평균 대여량)에서 `hour`와 `rented_count`는 각각 어느 쪽으로 다룹니까?

### 1.3 Series 만들기

`bike`의 **첫 24행**은 첫째 날의 24시간 기록입니다. 이 24행에서 대여량만 꺼내 `day1`로 저장하세요. 열 하나를 꺼낸 결과의 **타입**을 확인하고, 레이블 인덱스, 값의 개수, 형태, 자료형을 확인하세요. 레이블은 어떤 값입니까?

### 1.4 레이블과 절대 위치로 값 선택

`day1`에서 다음을 구하세요.

1. 절대 위치 5의 값과 레이블 18의 값
2. 절대 위치 6~8 범위와 레이블 6~8 범위에서 각각 **값이 몇 개**인지

그다음 둘째 날의 대여량 `day2`(행 24~47)를 만들고, **레이블 30**의 값과 **절대 위치 6**의 값을 비교하세요. 두 값이 같은 시간의 값입니까? 왜 이런 결과가 나옵니까?

### 1.5 Series의 조건 추출, 산술, 통계

`day1`에서 다음을 구하세요.

1. 대여량이 **500 이상**인 시간(레이블)과 값
2. 대여량이 **200 이상 500 미만**인 시간(레이블)
3. 평균, 중앙값, 최댓값, 그리고 최댓값이 나온 **시간**
4. 대여량을 **10% 늘린** 새로운 Series. 원래의 `day1`은 바뀌었습니까?

### 1.6 대입과 copy()

`day1`의 앞 세 값(위치 0~2)으로 새 Series `a`를 만들되, 원본에 영향이 없도록 복사본으로 만드세요. 그다음 `b = a`와 `c = a.copy()`를 만들고, `b`의 레이블 0 값을 0으로 바꾼 뒤 `a`, `b`, `c`의 레이블 0 값을 각각 출력하세요. 무엇이 바뀌었습니까? 분석에서 원본을 보존하는 방법을 한 줄로 쓰세요.

### 1.7 데이터프레임 확인

`bike`에는 무엇이 저장되어 있습니까? 파일 이름입니까, 표입니까? 자료형을 코드로 확인하고, 이 데이터에서 **관측치(행) 하나**와 **변수(열) 하나**가 각각 무엇을 나타내는지 설명하세요.

### 1.8 데이터 미리보기

데이터의 처음 5행, 마지막 3행, 무작위 표본 3행(`random_state=10`으로 고정)을 미리 보세요. 세 가지를 **모두** 확인하는 이유를 설명하세요.

### 1.9 데이터 크기와 변수 자료형

관측치(행)와 변수(열)의 수, 변수 이름, 변수별 자료형을 확인하세요. 수치형 변수 세 가지와 문자열(범주형) 변수 세 가지를 적으세요.

### 1.10 info()로 데이터 구조 확인

`info()`를 실행하고 `Non-Null Count`와 `Dtype`을 읽으세요. 비어 있는 값이 있는 열이 있습니까? `date`는 어떤 자료형이며, **이 질문에서** 날짜를 날짜형으로 바꿀 필요가 있습니까?

### 1.11 열 하나 꺼내기와 고유값

`bike`에서 열 하나(`season`)를 꺼내 **타입**을 확인하세요. 그다음 `season`, `holiday`, `functioning_day` 세 열이 각각 가진 **고유한 값**을 확인하세요. 계절은 몇 종류입니까?

### 1.12 행과 열을 함께 선택

`bike`의 행 0~2에서 `date`, `hour`, `rented_count` 세 열을 ① **열 이름**으로 ② **열 위치**로 각각 선택하세요. 열 위치로 고르려면 열 순서를 알아야 합니다. 두 결과에서 **행의 개수**와 **열의 순서**는 같습니까?

### 1.13 변수 선택과 필터링

질문에 필요한 변수 두 개만 선택해 새 변수로 저장하고 앞부분을 확인하세요. 그다음 **18시 관측치**만 필터링해 몇 건인지 세세요.

### 1.14 복합 조건 필터링

**18시이고 기온이 25도 이상인** 관측치가 몇 건인지 구하세요. 조건 두 개를 함께 쓸 때 코드에서 주의할 점을 한 줄로 쓰세요.

### 1.15 새로운 열 만들기

풍속(`wind_speed`, m/s)을 시속(km/h)으로 환산한 열 `wind_kmh`를 만드세요. 1 m/s는 3.6 km/h입니다. 원본 `bike`를 보존하기 위해 복사본 `bike2`에서 작업하고, 소수 첫째 자리까지 반올림하세요. 풍속이 가장 센 관측치 3개의 풍속과 환산값을 확인하세요.

### 1.16 정렬로 상위 관측치 확인

대여량이 **가장 큰** 관측치 5개를 날짜, 시간, 대여량, 계절과 함께 확인하세요. 어떤 공통점이 있습니까?

### 1.17 열 삭제

`bike2`에서 `dew_point`와 `visibility` 열을 삭제한 `slim`을 만들고, `bike2`와 `slim`의 열 개수를 비교하세요. 삭제한 뒤에도 `bike2`의 열이 그대로인 이유는 무엇입니까?

### 1.18 결측값 확인

변수별 결측값의 개수를 구하고, 데이터 전체의 합계도 구하세요.

### 1.19 중복값 확인

중복된 행이 몇 개인지 구하고, 있다면 중복된 행을 **모두** 표시하는 방법으로 확인하세요.

### 1.20 비정상 값 탐색

결측값도 중복값도 없습니다. 그런데 이 데이터를 그대로 분석에 사용해도 될까요? 대여량이 가장 **작은** 관측치부터 확인하세요. 비정상적이거나 설명이 필요한 값이 있습니까? 있다면 **몇 건인지**, **어떤 공통점이 있는지**를 코드로 확인하세요. (힌트 다른 변수와 함께 보면 단서가 있을 수 있습니다.)

### 1.21 데이터 전처리 방법 결정

이 관측치들의 전처리 방법을 두 가지로 **평균 대여량을 계산해 비교**한 뒤 결정하세요.

- 방법 A 모든 관측치를 그대로 사용
- 방법 B 문제가 되는 관측치를 제외하고 사용 (제외한 데이터를 `operating`이라는 이름으로 저장하세요)

**무엇을 제외했고 왜 그렇게 했는지** 전처리 기준을 글로 쓰세요. 이후 분석은 선택한 데이터로 합니다.

### 1.22 이상값 탐지 기준

1.16에서 대여량이 매우 큰 관측치를 확인했습니다. 이 값들을 **이상값으로 판정할 기준**을 두 가지로 세워 비교하세요. `operating`의 대여량으로 계산합니다.

- 기준 ① 3사분위수 + 1.5 × (3사분위수 − 1사분위수)
- 기준 ② 평균 + 2 × 표준편차

각 기준의 **상한값**과, 상한을 **초과하는 관측치의 수**를 구하세요. 초과한 관측치가 어느 시간대와 계절에 집중되어 있는지도 확인한 뒤, **이 관측치들을 제외해야 하는지** 판단하세요.

### 1.23 기술통계 요약

`operating`의 대여량을 요약하세요.

1. 개수, 평균, 표준편차, 사분위수, 최댓값을 한 번에 확인하세요.
2. 평균과 중앙값을 비교하고, 차이가 말해 주는 것을 쓰세요.
3. 계절과 공휴일 여부의 **빈도와 비율(%)**을 구하세요. 비율이 한쪽으로 치우친 변수가 있다면, 그것이 그룹 비교에 어떤 영향을 줄지 생각해 보세요.

### 1.24 그룹별 집계와 분석 질문 답하기

처음 질문에 답합니다.

1. 시간대별로 **관측치 수, 평균, 중앙값**을 집계하세요. 표본 크기가 충분한지 `count`로 확인하세요.
2. 평균이 가장 큰 시간대를 **코드로** 찾고, 상위 3개와 하위 3개 시간대를 확인하세요.
3. 1.21의 전처리 결정이 결론을 바꾸는지 확인하세요(민감도 확인). `operating`과 `bike`(전체)로 같은 집계를 해 **나란히** 놓고 비교하세요. 가장 많은 시간대가 달라집니까? 값은 얼마나 달라집니까?
4. 질문에 **한 문장**으로 답하세요.

# 2. 문제 정의에서 시작하는 분석

이번에는 **모호한 문제**가 주어집니다. 문제를 분석 질문으로 정의하고, **EDA로 데이터를 점검하며 분석 방법을 설계**합니다.

> 운영 담당자 "날씨가 자전거 이용에 영향을 주는 것 같습니다. 확인해 주세요."

데이터는 1에서 정리한 `operating`을 그대로 씁니다.

### 2.1 분석 질문 정의

위 요청을 **분석 질문**으로 바꾸세요. 질문의 다섯 요소(대상, 측정값, 기준, 계산, 범위)를 모두 쓰고, 날씨 변수 중 **어떤 변수를 기준으로** 비교할지 고르세요. 질문이 "영향을 준다(인과)"를 묻고 있지는 않은지 점검하세요.

### 2.2 데이터 적합성 점검

분석하기 전에 데이터가 질문에 답하기에 적합한지 **EDA로 점검**하세요. 내가 고른 기준으로 나눈 **그룹별 관측치 수**를 확인하고, 한쪽이 너무 적지는 않은지 보세요.

### 2.3 분석 방법 설계

바로 비교하기 전에, **단순 비교가 타당한지** 데이터로 확인하세요. 비가 오는 시간이 특정 계절에 편중되어 있다면, 계절이 비와 대여량 **양쪽에 영향을 주는 교란 요인**이 되어 비의 효과가 왜곡될 수 있습니다.

1. 계절마다 비 오는 시간의 **비율**을 구하세요.
2. 결과를 보고, **교란 요인을 어떻게 통제하며 비교할지** 분석 방법을 쓰세요.

### 2.4 비교 분석 실행

11.7에서 설계한 방법으로 비교하세요. 먼저 **단순 비교**(계절 구분 없음)를 하고, 그다음 **계절을 고정한 비교**를 하세요. 두 결과를 나란히 놓고, 비 오는 쪽 평균이 낮은 것이 **모든 계절에서** 나타나는지 확인하세요. 그룹의 **건수**도 반드시 함께 읽으세요.

### 2.5 결과 해석과 한계

세 가지로 나누어 쓰세요. **확인한 사실**(숫자 포함), **가능한 해석**(인과를 단정하지 않기), **한계**(이 결과가 말해 주지 못하는 것). 한계에는 **표본이 작은 그룹**과 **통제하지 못한 다른 조건**을 포함하세요.